# Exercise 3 — Self-Reflection Prompt for Improving Output

**Goal:** have the model write a summary, then critique its own summary against explicit requirements and produce a revised version (critique-and-revise). Programmatic checks verify the before/after.

**Tools:** Google Colab · Anthropic Python SDK (Claude) · Python checks for length/format/accuracy

**Flow**

```
source text → Prompt A (initial summary) → BEFORE
BEFORE + requirements → Prompt B (self-critique + revise) → AFTER
automated checks on BEFORE vs AFTER
```

## Setup

Uses the Anthropic Python SDK. Add your key to Colab **Secrets** (🔑 icon in the left sidebar) as `ANTHROPIC_API_KEY`, or paste it when prompted.

In [1]:
!pip install -q anthropic

import os, json, re, textwrap
from anthropic import Anthropic

try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    from getpass import getpass
    os.environ["ANTHROPIC_API_KEY"] = getpass("Anthropic API key: ")

client = Anthropic()
MODEL = "claude-sonnet-5"   # any current Claude model works; "claude-haiku-4-5" is cheaper/faster

def get_text(resp) -> str:
    """Join the text blocks of a response (newer models may also return thinking blocks)."""
    return "".join(b.text for b in resp.content if b.type == "text").strip()

def ask(system: str, user: str, max_tokens: int = 4000) -> str:
    """Single-turn call: system prompt + one user message -> text."""
    resp = client.messages.create(
        model=MODEL,
        max_tokens=max_tokens,
        system=system,
        messages=[{"role": "user", "content": user}],
        thinking={"type": "disabled"},   # the prompts themselves carry the reasoning steps
    )
    return get_text(resp)

def parse_json(text: str):
    """Tolerant JSON parse: strips ```json fences if the model adds them."""
    cleaned = re.sub(r"^```(?:json)?\s*|\s*```$", "", text.strip(), flags=re.S)
    return json.loads(cleaned)

def show(title: str, body: str):
    print(f"\n{'='*70}\n{title}\n{'='*70}\n{body}")

print("Ready. Model:", MODEL)

Ready. Model: claude-sonnet-5


## Source text
An original internal engineering update (written for this exercise), so accuracy can be checked against known numbers.

In [2]:
SOURCE = """Engineering update: checkout platform migration, Q3

Over the last quarter the payments team moved the checkout service off the legacy monolith and onto
three independently deployable services (cart, pricing, and payment-capture) running on Kubernetes, with
gRPC between them. The migration was completed on August 29, two weeks behind the original plan, mainly
because the pricing service needed a second round of load testing after it failed at 4,000 requests per
second in staging.

Results so far: p95 checkout latency dropped from 1.8 seconds to 0.9 seconds, and the deploy frequency
for checkout changes went from roughly once every two weeks to 14 deploys in the first two weeks of
September. Infrastructure cost for checkout rose by about 11% month over month, which we expect to
recover once the two remaining monolith nodes are decommissioned in October. There was one customer-facing
incident during cutover (a 23-minute window on August 29 where 3% of card payments returned a retry error);
all affected orders were reprocessed the same day and no revenue was lost.

Next quarter the team will retire the monolith nodes, add automated canary analysis to the deploy pipeline,
and begin migrating the refunds workflow, which is currently the largest remaining piece of the monolith."""

print(len(SOURCE.split()), "words")

204 words


## Prompt A — initial summary (the "before")
Deliberately loose, the way a first-draft prompt usually looks.

In [3]:
SUMMARY_SYSTEM = "You are a helpful assistant."
BEFORE = ask(SUMMARY_SYSTEM, f"Summarize this update:\n\n{SOURCE}", max_tokens=4000)
show("BEFORE — initial summary", BEFORE)


BEFORE — initial summary
## Summary

**What happened:** The payments team migrated the checkout service off the legacy monolith to three Kubernetes-based microservices (cart, pricing, payment-capture) communicating via gRPC. Completed August 29, two weeks late due to a pricing service failure at 4,000 RPS during load testing that required a retest.

**Results:**
- **Latency:** p95 checkout latency improved significantly (1.8s → 0.9s)
- **Deploy velocity:** Up sharply — from ~1 deploy/2 weeks to 14 deploys in early September
- **Cost:** Infrastructure spend up ~11% MoM, expected to normalize once remaining monolith nodes are decommissioned
- **Incident:** One 23-minute cutover incident (3% of card payments got retry errors); fully remediated same-day with no revenue loss

**Next steps (Q4):**
1. Decommission remaining monolith nodes
2. Add automated canary analysis to the deploy pipeline
3. Begin migrating the refunds workflow (largest remaining monolith component)

**Bottom line:** A 

## Requirements the critique must check
These are the explicit criteria; they are used both in the reflection prompt and in the automated checks below.

In [4]:
REQUIREMENTS = """1. AUDIENCE: a non-technical VP. No engineering jargon — specifically avoid: Kubernetes, gRPC, p95, canary, monolith, staging, nodes.
2. LENGTH: 60 words or fewer in total.
3. FORMAT: exactly 3 bullet points, each starting with "- ". No heading, no intro sentence.
4. ACCURACY: every number must appear in the source text exactly; do not round, convert, or add numbers.
5. FOCUS: bullet 1 = business outcome (speed / shipping cadence), bullet 2 = cost and risk (incident), bullet 3 = what happens next.
6. CLARITY: lead each bullet with the outcome, not the activity."""

JARGON = ["kubernetes", "grpc", "p95", "canary", "monolith", "staging", "nodes"]

def check(summary: str) -> dict:
    bullets = [l for l in summary.strip().splitlines() if l.strip().startswith("- ")]
    nums_out = set(re.findall(r"\d+(?:[.,]\d+)?%?", summary))
    nums_src = set(re.findall(r"\d+(?:[.,]\d+)?%?", SOURCE))
    return {
        "word_count": len(summary.split()),
        "within_60_words": len(summary.split()) <= 60,
        "exactly_3_bullets": len(bullets) == 3 and len(summary.strip().splitlines()) == 3,
        "no_jargon": not any(j in summary.lower() for j in JARGON),
        "numbers_all_in_source": nums_out <= nums_src,
        "unsupported_numbers": sorted(nums_out - nums_src),
    }

print(json.dumps(check(BEFORE), indent=2))

{
  "word_count": 161,
  "within_60_words": false,
  "exactly_3_bullets": false,
  "no_jargon": false,
  "numbers_all_in_source": false,
  "unsupported_numbers": [
    "1",
    "2",
    "4"
  ]
}


## Prompt B — self-reflection: critique, then revise

### Iteration: v1 (generic) → v2 (explicit criteria)
v1 just said "review and improve". It produced a slightly shorter paragraph with no clear reason for the changes. v2 gives the model the checklist, a pass/fail verdict per criterion, and a required output format.

In [5]:
# v1 (kept for comparison)
REFLECT_V1 = "Review your summary and improve it."
show("v1 reflection output (weak criteria)", ask("You are a helpful assistant.",
     f"Source:\n{SOURCE}\n\nYour summary:\n{BEFORE}\n\n{REFLECT_V1}", max_tokens=4000))


v1 reflection output (weak criteria)
## Summary

**What happened:** The payments team migrated the checkout service off the legacy monolith to three independently deployable services (cart, pricing, payment-capture) running on Kubernetes with gRPC communication. Completed August 29 — two weeks behind schedule after the pricing service failed load testing at 4,000 requests/sec and required a second round of testing.

**Results:**
- **Latency:** p95 checkout latency nearly halved (1.8s → 0.9s)
- **Deploy velocity:** Rose from ~1 deploy every 2 weeks to 14 deploys in the first two weeks of September
- **Cost:** Infrastructure spend up ~11% month-over-month; expected to come down once the two remaining monolith nodes are decommissioned in October
- **Incident:** A 23-minute window during cutover on August 29 caused retry errors on 3% of card payments; all affected orders were reprocessed same-day with no revenue lost

**Next steps (Q4):**
1. Decommission the two remaining monolith nodes
2

In [6]:
REFLECT_SYSTEM = """You are an editor reviewing a summary you wrote earlier. Be strict and specific.

Step 1 — CRITIQUE: check the summary against every requirement below. For each, write PASS or FAIL
with a one-line reason and, on FAIL, quote the offending phrase.
Step 2 — REVISE: write a new summary that satisfies ALL requirements. Check it again yourself
(count the words, count the bullets, verify every number against the source) before you output it.

Output format, exactly (no text outside the tags):
<critique>
1. AUDIENCE: PASS|FAIL — <reason>
2. LENGTH: PASS|FAIL — <word count>
3. FORMAT: PASS|FAIL — <reason>
4. ACCURACY: PASS|FAIL — <reason>
5. FOCUS: PASS|FAIL — <reason>
6. CLARITY: PASS|FAIL — <reason>
</critique>
<revised>
- ...
- ...
- ...
</revised>
"""

def reflect_and_revise(summary: str) -> tuple[str, str]:
    user = (f"REQUIREMENTS:\n{REQUIREMENTS}\n\nSOURCE TEXT:\n{SOURCE}\n\n"
            f"SUMMARY TO REVIEW:\n{summary}")
    text = ask(REFLECT_SYSTEM, user, max_tokens=4000)
    critique = re.search(r"<critique>(.*?)</critique>", text, re.S).group(1).strip()
    revised  = re.search(r"<revised>(.*?)</revised>", text, re.S).group(1).strip()
    return critique, revised

critique, AFTER = reflect_and_revise(BEFORE)
show("SELF-CRITIQUE", critique)
show("AFTER — revised summary", AFTER)


SELF-CRITIQUE
1. AUDIENCE: FAIL — contains "Kubernetes", "gRPC", "monolith", "p95", "staging", "canary", "nodes" throughout.
2. LENGTH: FAIL — roughly 180+ words, far exceeds 60 word limit.
3. FORMAT: FAIL — has headings ("## Summary", "**What happened:**"), numbered list, and intro sentences instead of exactly 3 bullets starting with "- ".
4. ACCURACY: PASS — numbers used (1.8s, 0.9s, 14 deploys, 11%, 23-minute, 3%) match source exactly.
5. FOCUS: FAIL — includes extra sections (What happened, Bottom line) beyond the required 3-bullet structure of outcome/cost/next steps.
6. CLARITY: FAIL — bullets lead with activity labels ("Latency:", "Deploy velocity:") rather than outcomes.

AFTER — revised summary
- Checkout releases are now much faster to ship — 14 updates in two weeks, up from one every two weeks, with load times cut in half (1.8s to 0.9s).
- A brief August 29 issue caused retry errors for 3% of card payments; fully fixed same day, no revenue lost. Costs rose ~11%, expected to

## Verify: before vs after
If the revision still fails a check, run one more reflection pass on it (the loop is the point of critique-and-revise).

In [7]:
for attempt in range(2):
    result = check(AFTER)
    if all(v for k, v in result.items() if k not in ("word_count", "unsupported_numbers")):
        break
    print(f"Revision still fails {[k for k,v in result.items() if v is False]} — running another pass")
    critique2, AFTER = reflect_and_revise(AFTER)
    show(f"SELF-CRITIQUE (pass {attempt+2})", critique2)

show("BEFORE", BEFORE + "\n\nchecks: " + json.dumps(check(BEFORE)))
show("AFTER",  AFTER  + "\n\nchecks: " + json.dumps(check(AFTER)))

Revision still fails ['within_60_words'] — running another pass



SELF-CRITIQUE (pass 2)
1. AUDIENCE: FAIL — "load times cut in half (1.8s to 0.9s)" implies p95 latency jargon carried over via precise metric framing, and "canary" appears as "automated safety checks" (fine) but "decommissioning" is jargon-adjacent; more critically no banned terms literally appear, so this is borderline PASS — reconsider: no banned words actually used. PASS — no listed banned terms appear.
2. LENGTH: FAIL — word count is 76 words, exceeds 60-word limit.
3. FORMAT: PASS — exactly 3 bullets, each starts with "- ", no heading or intro.
4. ACCURACY: FAIL — "1.8s to 0.9s" converts "1.8 seconds to 0.9 seconds" to abbreviated units not in source; "~11%" alters "about 11%" wording though number preserved, borderline; main issue is unit conversion.
5. FOCUS: FAIL — bullet 2 mixes incident with cost/risk correctly, but bullet 1 doesn't lead with pure business outcome clearly separated from activity description ("14 updates" is fine); actually focus mapping roughly holds but bul

## Notes
- **Measured, not eyeballed:** BEFORE was 161 words with headings, jargon and a numbered list — it failed 4 of 5 checks. The final AFTER is 58 words, exactly 3 bullets, no jargon, every number traceable to the source.
- **The loop mattered:** the first revision fixed audience, format and focus but was still over 60 words, so the notebook ran a second reflection pass, which cut it to 58 and also dropped the "1.8s" unit abbreviation it had flagged as an accuracy risk.
- **v1 vs v2 reflection:** "Review your summary and improve it" produced a near-identical long summary plus a note about its own edits — no criteria, no measurable change. The v2 prompt (explicit PASS/FAIL per criterion + required format) produced the targeted revision.
- **Fixes while building:** the first run crashed because the model spent its entire token budget thinking before writing anything; disabling thinking fixed it. The output format was also switched from markdown headers to `<critique>`/`<revised>` tags, which are more reliable to parse.